# Stages 8–9: align all sections
Reads: the VAD sections, their API transcripts, the book.
Writes: data/test/alignment.json — every spoken word, once, with its time in the whole recording, its score and how much was trimmed.

After changing anything in `scripts/`, restart the kernel and run all cells.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "../../scripts")

from paths import SECTIONS_DIR, SECTIONS_JSON, RAW_RESPONSES, BOOK_TEXT, ALIGNMENT_JSON
from segments import join_segments
from sections import load_sections, to_recording_time, keep_owned
from uzbek_text import fix_apostrophes
from aligner import load_model, align

In [ ]:
model = load_model()

In [ ]:
sections = load_sections(SECTIONS_JSON)
book_words = BOOK_TEXT.read_text(encoding="utf-8").split()

for section in sections:
    name = Path(section["filename"]).stem
    response_path = RAW_RESPONSES / f"{name}.response"
    response = json.loads(response_path.read_text(encoding="utf-8"))

    text = join_segments(response["segments"], book_words)
    section["text"] = fix_apostrophes(text)
    print(f"{name}: {len(section['text'].split())} words")

In [ ]:
all_words = []

for section in sections:
    audio_path = SECTIONS_DIR / section["filename"]
    words = align(model, audio_path, section["text"])

    words = to_recording_time(words, section)
    owned = keep_owned(words, section)

    for word in owned:
        word["section"] = section["filename"]

    all_words.extend(owned)
    print(f"{section['filename']}: aligned {len(words)}, kept {len(owned)}")

print()
print("Total words:", len(all_words))

In [ ]:
out_of_order = 0
for before, after in zip(all_words, all_words[1:]):
    if after["start"] < before["start"]:
        out_of_order += 1
print("Words out of time order:", out_of_order)
print()

for before, after in zip(sections, sections[1:]):
    last_words = [w["word"] for w in all_words if w["section"] == before["filename"]][-4:]
    first_words = [w["word"] for w in all_words if w["section"] == after["filename"]][:4]
    print(f"{before['own_end']:7.2f} s:  ... {' '.join(last_words)} | {' '.join(first_words)} ...")
print()

for word in all_words:
    if word["end"] - word["start"] > 2:
        print("Long word:", word)

In [ ]:
trimmed = [word for word in all_words if word["trimmed"] > 0.3]
print("Words trimmed by more than 0.3 s:", len(trimmed))
for word in trimmed:
    print(f"   {word['word']:15} {word['start']:7.2f}-{word['end']:7.2f}  trimmed {word['trimmed']:5.2f} s  score {word['score']}")

print()
print("The 10 words the model was least sure about:")
for word in sorted(all_words, key=lambda w: w["score"])[:10]:
    print(f"   {word['score']:6.2f}  {word['word']:15} {word['start']:7.2f}-{word['end']:7.2f}  {word['section']}")

scores = sorted(word["score"] for word in all_words)
print()
print("Middle score:", scores[len(scores) // 2])

In [ ]:
ALIGNMENT_JSON.write_text(
    json.dumps(all_words, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Saved", len(all_words), "words to", ALIGNMENT_JSON)